# Electroporation

{class}`~pylabrobot.capabilities.electroporation.electroporation.Electroporation` prepares, starts, and cancels electroporation runs.

The standard workflow is a prepared run: first create a temporary protocol on the device and leave the instrument armed on its run screen, then start that prepared run when the labware and safety interlocks are ready.

## Walkthrough

This example uses a chatterbox backend so the workflow can be run without hardware.

In [ ]:
from pylabrobot.capabilities.electroporation import (
  Electroporation,
  ElectroporationChatterboxBackend,
  ElectroporationProtocol,
)

electroporator = Electroporation(backend=ElectroporationChatterboxBackend())
await electroporator._on_setup()

Define a protocol. Square-wave protocols use `duration_us`; exponential-decay protocols use `resistance_ohms` and `capacitance_uf`.

In [ ]:
protocol = ElectroporationProtocol(
  protocol_type="square",
  pulse_amplitude_volts=250,
  gap_mm=2.0,
  pulse_count=1,
  pulse_interval_seconds=0.0,
  duration_us=1000,
)

Prepare the run. On real hardware this writes a temporary protocol and leaves the device on the run screen. `plate_columns` is used by high-throughput plate-handler workflows.

In [ ]:
prepared = await electroporator.prepare_temporary_protocol(
  protocol=protocol,
  plate_columns=3,
)
prepared.protocol_name

Prepared runs are serializable so a later process can resume from the same armed state.

In [ ]:
prepared_payload = prepared.as_dict()
prepared_payload["protocol_name"]

Start the prepared run and collect the result. Hardware backends may include device logs and cleanup information in the result.

In [ ]:
result = await electroporator.start_prepared_run(prepared)
result.log_capture.summary

To arm a protocol but then stop before pulse delivery, cancel the prepared run.

In [ ]:
prepared = await electroporator.prepare_temporary_protocol(protocol=protocol)
cancelled = await electroporator.cancel_prepared_run(prepared)
cancelled.cleanup.deleted

## Supported hardware

```{supported-devices} electroporation
```

## API reference

See {class}`~pylabrobot.capabilities.electroporation.electroporation.Electroporation`, {class}`~pylabrobot.capabilities.electroporation.backend.ElectroporationBackend`, and {class}`~pylabrobot.capabilities.electroporation.standard.ElectroporationProtocol`.